# Paleoclimate model comparison data exploration

Four paleoclimate temperature datasets (Beyer2020, PalMod2, TraCE-21k,
CHELSA-TraCE21k) are inspected here. The notebook produces:

1. a per-dataset text report (attrs, dims, coords, time convention, the
   temperature variable, the spatial grid);
2. a 2-panel visual summary (temporal coverage + which grid cell each dataset
   returns near a point);
3. a comparison plot of air-temperature time series at one location;
4. a world map of difference heatmaps between dataset pairs.

The hard part is the **time axis** — every file uses a different convention.
`time_to_ka_bp` centralises that translation so the rest of the notebook can
treat all four uniformly.

## 1. Imports

All third-party imports live here so each function cell stays focused.

In [ ]:
# imports
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

## 2. Configuration

All the constants the functions below share, gathered in one place so a new
dataset (or a rotated grid that uses `rlat`/`rlon`) only needs an edit here.

In [ ]:
# Coordinate-name candidates. The first match found in a file's coords wins.
LAT_NAMES = ["lat", "latitude", "nav_lat", "y"]
LON_NAMES = ["lon", "longitude", "nav_lon", "x"]

In [ ]:
# Depth/soil-layer dimension names. When present, we keep the topmost layer.
DEPTH_NAMES = ["levsoi", "depth"]

In [ ]:
# Dataset colors
DATASET_COLORS = {
    "Beyer2020":                  "steelblue",
    "PalMod2":                    "darkorange",
    "TraCE-21k":                  "seagreen",
    "CHELSA-TraCE21k-centennial": "indianred",
}
DEFAULT_COLOR = "grey"  # fallback for any dataset not in the dict

## 3. Data file paths

Paths are resolved relative to the working directory, so the notebook runs on
any machine that mirrors the `data/` layout.

In [ ]:
DATA = Path.cwd() / "data"

beyer2020_air = str(DATA / "Beyer2020"                  / "data"   / "LateQuaternary_Environment.nc")
palmod2_air   = str(DATA / "PalMod2"                    / "output" / "PMMXMCRTDGr111Amtasgn30201_1-250.nc")
trace_air     = str(DATA / "TraCE-21k"                  / "data"   / "trace.01-36.22000BP.clm2.TSA.22000BP_decavg_400BCE.nc")
chelsa_air    = str(DATA / "CHELSA-TraCE21k-centennial" / "output" / "tasmean.nc")

palmod2_soil  = str(DATA / "PalMod2"                    / "output" / "PMMXMCRTDGr111Lmtslgn30201_1-250.nc")
trace_soil    = str(DATA / "TraCE-21k"                  / "data"   / "trace.01-36.22000BP.clm2.TSOI.22000BP_decavg_400BCE.nc")

## 4. Dataset registries

Two DataFrames the rest of the notebook iterates over. 
One for near-surface air temperature, one for soil temperature. Columns:

- `name`: short label for plots and reports
- `path`: absolute path to the NetCDF file
- `variable`: name of the temperature variable inside the file
- `unit`: `"degC"` or `"K"` (drives the K->°C conversion downstream)

In [ ]:
df_air = pd.DataFrame(
    [
        ["Beyer2020",                  beyer2020_air, "temperature", "degC"],
        ["PalMod2",                    palmod2_air,   "tas",         "K"],
        ["TraCE-21k",                  trace_air,     "TSA",         "K"],
        ["CHELSA-TraCE21k-centennial", chelsa_air,    "tasmean",     "K"],
    ],
    columns=["name", "path", "variable", "unit"],
)
df_air

In [ ]:
df_soil = pd.DataFrame(
    [
        ["PalMod2",   palmod2_soil, "tsl",  "K"],
        ["TraCE-21k", trace_soil,   "TSOI", "K"],
    ],
    columns=["name", "path", "variable", "unit"],
)
df_soil

## 5. Time conversion: NetCDF time coord -> ka BP

Returns **ka BP with past = positive, present = 0** for any of the four files.

| Dataset    | `units` attribute              | raw range          | conversion to ka BP     |
|------------|--------------------------------|--------------------|-------------------------|
| Beyer2020  | `years since present`          | `[-120000, 0]`     | `-vals / 1000`          |
| PalMod2    | `days since 1-1-1 00:00:00`    | `[181, 9 130 878]` | `(max - vals) / 365250` |
| TraCE-21k  | `ka BP` (but values negative!) | `[-22, 0.03]`      | `-vals`                 |
| CHELSA     | `days since -20010-07-01 ...`  | `[0, 8 035 335]`   | `(max - vals) / 365250` |

For the `days since <date>` files the absolute reference date is irrelevant.
We only need "how long before present", so we anchor on `vals.max()` (the most recent sample). 
`365250 = 1000 × 365.25` days = 1 kyr. 
The ~0.5 ka rounding this introduces is invisible on a 0–125 ka axis.

In [ ]:
def time_to_ka_bp(time_da):
    """Convert a NetCDF time coord to ka BP (past = positive, present = 0)."""
    units = (time_da.attrs.get("units") or "").lower()
    vals = np.asarray(time_da.values, dtype=float)

    # TraCE: "ka BP" but values are negative kiloyears.
    if "ka" in units or "kyr" in units:
        return -vals if vals.min() < 0 else vals

    # Beyer: "years since present": past is negative.
    if "since present" in units:
        return -vals / 1000

    # PalMod2, CHELSA: "days since <date>": anchor on the last sample.
    if "since" in units and "day" in units:
        return (vals.max() - vals) / 365250.0

    # Fallback: the sign of the data tells the convention.
    return (-vals / 1000) if vals.min() < 0 else (vals / 1000)

## 6. Dataset inspection

`inspect_dataset` returns a multi-line string describing one NetCDF file:
global attrs, dims, coords, time convention (raw + converted via `time_to_ka_bp`), 
the temperature variable, and the spatial grid.

In [ ]:
def inspect_dataset(name, path, variable, unit_in_df):
    """Return a multi-line description of one NetCDF dataset."""
    lines = [
        "=" * 78,
        f"DATASET: {name}",
        "=" * 78,
        f"path     : {path}",
        f"variable : {variable}   (declared unit in df_air: {unit_in_df})",
        "",
    ]

    with xr.open_dataset(path, decode_times=False) as ds:

        # global attributes
        lines.append("global attributes")
        if ds.attrs:
            for k, v in ds.attrs.items():
                lines.append(f"  {k}: {v}")
        else:
            lines.append("  (none)")
        lines.append("")

        # dimensions
        # ds.sizes (not ds.dims) avoids a FutureWarning in xarray >= 2024.
        lines.append("dimensions")
        for d, n in ds.sizes.items():
            lines.append(f"  {d}: {n}")
        lines.append("")

        # coordinates
        lines.append("coordinates")
        for cname in ds.coords:
            c = ds[cname]
            vals = np.asarray(c.values)
            try:
                rng = f"[{float(vals.min()):.4f}, {float(vals.max()):.4f}]"
            except (TypeError, ValueError):
                # non-numeric coord (e.g. cftime) — fall back to endpoints
                rng = f"first={vals.flat[0]}, last={vals.flat[-1]}"
            lines.append(f"  {cname}  shape={c.shape}  dtype={c.dtype}  range={rng}")
            for ak, av in c.attrs.items():
                lines.append(f"      {ak}: {av}")
        lines.append("")

        # time convention (raw + converted)
        if "time" in ds.coords:
            t = ds["time"]
            tvals = np.asarray(t.values, dtype=float)
            ka = time_to_ka_bp(t)
            step = np.diff(tvals)
            step_info = (
                f"median = {np.median(step):.3f}, "
                f"min = {step.min():.3f}, max = {step.max():.3f}"
                if step.size else "n/a"
            )
            lines += [
                "time convention",
                f"  units              : {t.attrs.get('units', '<missing>')}",
                f"  raw range          : [{tvals.min():.3f}, {tvals.max():.3f}]  (n={tvals.size})",
                f"  raw step           : {step_info}",
                f"  converted ka BP    : [{float(ka.min()):.3f}, {float(ka.max()):.3f}]",
                "",
            ]

        # temperature variable
        lines.append(f"variable: {variable}")
        if variable in ds.variables:
            v = ds[variable]
            lines.append(f"  dims    : {v.dims}")
            lines.append(f"  shape   : {v.shape}")
            lines.append(f"  dtype   : {v.dtype}")
            for ak, av in v.attrs.items():
                lines.append(f"  attr {ak}: {av}")
        else:
            lines.append(f"  !! variable {variable!r} NOT FOUND in file.")
            lines.append(f"  available data vars: {list(ds.data_vars)}")
        lines.append("")

        # spatial grid (lat/lon detection mirrors the extraction logic)
        lat_name = next((n for n in LAT_NAMES if n in ds.coords), None)
        lon_name = next((n for n in LON_NAMES if n in ds.coords), None)
        lines.append("spatial grid")
        lines.append(f"  detected lat coord: {lat_name}")
        lines.append(f"  detected lon coord: {lon_name}")
        if lat_name and lon_name:
            la = ds[lat_name].values
            lo = ds[lon_name].values
            lon_conv = "0-360" if float(lo.max()) > 180 else "-180-180"
            lat_res = float(np.median(np.abs(np.diff(la))))
            lon_res = float(np.median(np.abs(np.diff(lo))))
            lines.append(
                f"  lat range : [{float(la.min()):.3f}, {float(la.max()):.3f}]"
                f"  resolution ~ {lat_res:.4f} deg"
            )
            lines.append(
                f"  lon range : [{float(lo.min()):.3f}, {float(lo.max()):.3f}]"
                f"  resolution ~ {lon_res:.4f} deg  (convention: {lon_conv})"
            )
        lines.append("")

    return "\n".join(lines)

### Run the inspection and save the report

Writes one `=`-separated block per dataset to `dataset_info.txt`.

In [ ]:
# PalMod2 and TraCE-21k overlap completely (grid cells).

# out_path = Path("dataset_info.txt")
# report = [inspect_dataset(r["name"], r["path"], r["variable"], r["unit"])
#           for _, r in df_air.iterrows()]
#
# out_path.write_text("\n".join(report), encoding="utf-8")
# print("\n".join(report))
# print(f"\n-> Saved full report to: {out_path.resolve()}")

## 7. Temperature extraction at a point

`extract_temperature_at_point` opens one file, picks the grid cell nearest a requested lat/lon, 
collapses the seasonal cycle (`month`) and depth if present,
converts K->°C if needed, and returns a 1-D series in ka BP coords.

In [ ]:
def extract_temperature_at_point(path, temp_var, lat, lon, input_unit="degC"):
    """Return (time_ka, y_degC, cell_lat, cell_lon) for one NetCDF file.

    Picks the topmost soil layer if a depth dim is present.
    """
    with xr.open_dataset(path, decode_times=False) as ds:
        lat_name = next(n for n in LAT_NAMES if n in ds.coords)
        lon_name = next(n for n in LON_NAMES if n in ds.coords)

        # Handle 0-360 vs -180-180 longitude conventions.
        lon_q = lon + 360 if float(ds[lon_name].max()) > 180 and lon < 0 else lon

        da = ds[temp_var].sel({lat_name: lat, lon_name: lon_q}, method="nearest")
        cell_lat = float(da[lat_name])
        cell_lon = float(da[lon_name])

        # Collapse extra dims: topmost soil layer, then annual mean.
        for d in DEPTH_NAMES:
            if d in da.dims:
                da = da.isel({d: 0})
                break
        if "month" in da.dims:
            da = da.mean(dim="month")
        if input_unit == "K":
            da = da - 273.15

        return time_to_ka_bp(ds["time"]), da.values.copy(), cell_lat, cell_lon

## 8. Comparison plot

Overlays the temperature series from every row of a registry on one axis. The
symlog x-axis expands the Holocene so recent change is visible next to the full
glacial cycle.

In [ ]:
def plot_temperature_comparison(df, lat, lon, var=None, log_x=True):
    """Overlay temperature time series from several NetCDF files on one axis."""
    fig, ax = plt.subplots(figsize=(16, 4))
    results = {}

    for _, row in df.iterrows():
        time_ka, y, cell_lat, cell_lon = extract_temperature_at_point(
            row["path"], row["variable"], lat, lon, input_unit=row["unit"],
        )
        ax.plot(
            time_ka, y,
            color=DATASET_COLORS.get(row["name"], DEFAULT_COLOR),
            lw=1.8, alpha=0.7,
            label=f"{row['name']}  ({cell_lat:.2f}N, {cell_lon:.2f}E)",
        )
        results[row["name"]] = (time_ka, y)

    if log_x:
        ax.set_xscale("symlog", linthresh=0.1)
        ax.set_xticks([0, 0.1, 1, 10, 100])
        ax.set_xticklabels(["0", "0.1", "1", "10", "100"])

    ax.invert_xaxis()
    ax.axhline(0, color="0.3", lw=0.6, ls="--", alpha=0.5)
    ax.set_xlabel("Time (ka BP)")
    ax.set_ylabel("Temperature (C)")
    ax.set_title(f"{var} temperature at ({lat:.2f}N, {lon:.2f}E)")
    ax.grid(alpha=0.25, which="both", linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, loc="best")
    plt.tight_layout()
    plt.show()
    return results

## 9. Visual summary (coverage + grid cells)

Two panels: temporal coverage (left) and the grid cell each dataset returns near the requested point (right). 
`collect_summary` does the data extraction so both panels (and the text report) cannot disagree.

In [ ]:
def collect_summary(df, lat_req, lon_req):
    """Minimal per-dataset info needed for the summary plot."""
    rows = []
    for _, r in df.iterrows():
        with xr.open_dataset(r["path"], decode_times=False) as ds:
            # Time: same conversion the comparison plot uses.
            ka = np.sort(time_to_ka_bp(ds["time"]))
            step_kyr = float(np.median(np.diff(ka))) if ka.size > 1 else np.nan

            # Spatial: which cell would .sel(method="nearest") pick?
            lat_name = next(n for n in LAT_NAMES if n in ds.coords)
            lon_name = next(n for n in LON_NAMES if n in ds.coords)

            # Handle 0-360 vs -180-180 longitude conventions.
            lon_q = lon_req + 360 if float(ds[lon_name].max()) > 180 and lon_req < 0 else lon_req

            sub = ds[r["variable"]].sel(
                {lat_name: lat_req, lon_name: lon_q}, method="nearest"
            )
            cell_lat = float(sub[lat_name])
            cell_lon = float(sub[lon_name])
            if cell_lon > 180:                    # show in -180..180 for plotting
                cell_lon -= 360

            lat_res = float(np.median(np.abs(np.diff(ds[lat_name].values))))
            lon_res = float(np.median(np.abs(np.diff(ds[lon_name].values))))

        rows.append({
            "name": r["name"],
            "t_min": float(ka.min()), "t_max": float(ka.max()),
            "n_steps": int(ka.size), "step_kyr": step_kyr,
            "cell_lat": cell_lat, "cell_lon": cell_lon,
            "lat_res": lat_res, "lon_res": lon_res,
        })
    return rows

In [ ]:
def plot_dataset_summary(df, lat_req, lon_req):
    """Side-by-side temporal coverage (left) and regional grid cells (right)."""
    rows = collect_summary(df, lat_req, lon_req)

    # constrained_layout handles cartopy axes correctly (tight_layout doesn't).
    fig = plt.figure(figsize=(16, max(5, 0.55 * len(rows) + 3)),
                     constrained_layout=True)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.4, 1])
    ax_t = fig.add_subplot(gs[0, 0])
    ax_m = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())

    # Panel 1: temporal coverage
    for i, r in enumerate(rows):
        c = DATASET_COLORS.get(r["name"], DEFAULT_COLOR)
        ax_t.hlines(i, r["t_min"], r["t_max"], colors=c, lw=7, alpha=0.85)
        ax_t.text(
            r["t_max"], i + 0.25,
            f"  dt ~ {r['step_kyr']*1000:.0f} yr   n={r['n_steps']}",
            va="bottom", ha="left", fontsize=9, color="0.4",
        )
    ax_t.set_yticks(range(len(rows)))
    ax_t.set_yticklabels([r["name"] for r in rows])
    # ax_t.set_xscale("symlog", linthresh=0.1)
    # ax_t.set_xticks([0, 0.1, 1, 10, 100])
    # ax_t.set_xticklabels(["0", "0.1", "1", "10", "100"])
    ax_t.set_xlim(150, -0.1)
    ax_t.invert_yaxis()
    ax_t.set_xlabel("Time (ka BP)")
    ax_t.set_title("Temporal coverage")
    ax_t.grid(alpha=0.25, which="both", axis="x", linewidth=0.5)
    ax_t.spines["top"].set_visible(False)
    ax_t.spines["right"].set_visible(False)

    # Panel 2: regional map
    pc = ccrs.PlateCarree()
    MARGIN = 5
    ax_m.set_extent(
        [lon_req - MARGIN, lon_req + MARGIN,
         lat_req - MARGIN, lat_req + MARGIN],
        crs=pc,
    )
    ax_m.add_feature(cfeature.LAND,      facecolor="#EAE6DE")
    ax_m.add_feature(cfeature.OCEAN,     facecolor="#D6E4F0")
    ax_m.add_feature(cfeature.COASTLINE, lw=0.4, edgecolor="0.4")
    ax_m.add_feature(cfeature.BORDERS,   lw=0.3, edgecolor="0.5", alpha=0.6)
    ax_m.gridlines(lw=0.3, color="0.7", alpha=0.5, draw_labels=True)

    # Each rectangle is one dataset's grid cell, drawn at its true size in degrees.
    for r in rows:
        c = DATASET_COLORS.get(r["name"], DEFAULT_COLOR)
        ax_m.add_patch(Rectangle(
            (r["cell_lon"] - r["lon_res"] / 2, r["cell_lat"] - r["lat_res"] / 2),
            r["lon_res"], r["lat_res"],
            edgecolor=c, facecolor=c, alpha=0.35, lw=1.4,
            transform=pc, zorder=3,
            label=f"{r['name']}  ({r['lat_res']:.2f}x{r['lon_res']:.2f} deg)",
        ))
        ax_m.scatter([r["cell_lon"]], [r["cell_lat"]],
                     color=c, s=35, zorder=4, alpha=0.9, transform=pc)

    ax_m.scatter([lon_req], [lat_req], marker="*", s=260,
                 color="crimson", edgecolor="k", lw=0.6,
                 transform=pc, zorder=5, label="requested")

    ax_m.set_title(f"Grid cells near ({lat_req:.2f}N, {lon_req:.2f}E)")
    ax_m.legend(fontsize=8, loc="lower left", frameon=False)

    plt.show()
    return rows

## 10. Points of interest

`poi` is the location actually plotted. `cities` is just for testing purposes.

In [ ]:
poi = {"022_00TG": (53.25563076095137, 11.672755742220097)}

cities = {
    # "Berlin":      (52.52,  13.40),
    # "Paris":       (48.85,   2.35),
    # "Moscow":      (55.75,  37.62),
    # "Cape Town":  (-33.92,  18.42),
    # "New York":    (40.71, -74.01),
    # "Madrid":      (40.42,  -3.70),
    # "Los Angeles": (34.05,-118.24),
    # "Kampala":     ( 0.32,  32.58),
    # "Mumbai":      (19.08,  72.88),
    # "Bangkok":     (13.76, 100.50),
    # "Tokyo":       (35.68, 139.65),
    # "Helsinki":    (60.17,  24.94),
    # "Oslo":        (59.91,  10.75),
    # "Reykjavik":   (64.15, -21.94),
}

### Run air-temperature plots

In [ ]:
air_results = {}
air_summaries = {}
for name, (lat, lon) in poi.items():
    print(name)
    air_results[name]   = plot_temperature_comparison(df_air, lat=lat, lon=lon, log_x=False, var="Air")
    air_summaries[name] = plot_dataset_summary(df_air, lat_req=lat, lon_req=lon)

### Run soil-temperature plots

In [ ]:
soil_results = {}
soil_summaries = {}
for name, (lat, lon) in poi.items():
    print(name)
    soil_results[name]   = plot_temperature_comparison(df_soil, lat=lat, lon=lon, log_x=False, var="Soil")
    soil_summaries[name] = plot_dataset_summary(df_soil, lat_req=lat, lon_req=lon)

## 11. Grid utilities

`grid_cell_centers` lists every cell center of a file (a drop-in replacement
for `cities`); `filter_europe` trims to a Europe bounding box;
`plot_grid_on_worldmap` scatters the centers onto a world map.

In [ ]:
def grid_cell_centers(path):
    """{label: (lat, lon)} for every grid cell of one file."""
    with xr.open_dataset(path, decode_times=False) as ds:
        lat_name = next(n for n in LAT_NAMES if n in ds.coords)
        lon_name = next(n for n in LON_NAMES if n in ds.coords)
        lats = [float(v) for v in ds[lat_name].values]
        lons = [float(v) for v in ds[lon_name].values]

    centers = {}
    for la in lats:
        for lo in lons:
            lo_pm = lo - 360 if lo > 180 else lo   # 0-360 -> -180-180, matches cities
            centers[f"({la:.2f}, {lo_pm:.2f})"] = (la, lo_pm)
    return centers

In [ ]:
def plot_grid_on_worldmap(centers, title="Grid cell centers"):
    """Scatter every (lat, lon) from a centers dict onto a world map."""
    lats = [la for la, lo in centers.values()]
    lons = [lo for la, lo in centers.values()]

    fig = plt.figure(figsize=(14, 7))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    ax.set_global()
    ax.add_feature(cfeature.LAND,      facecolor="#EAE6DE")
    ax.add_feature(cfeature.OCEAN,     facecolor="#D6E4F0")
    ax.add_feature(cfeature.COASTLINE, lw=0.4, edgecolor="0.4")
    ax.gridlines(lw=0.3, color="0.7", alpha=0.5, draw_labels=True)

    ax.scatter(lons, lats, s=4, color="crimson", alpha=0.6,
               transform=ccrs.PlateCarree(), zorder=3)

    ax.set_title(f"{title}  (n={len(centers)})")
    plt.tight_layout()
    plt.show()


def filter_europe(centers, lon_min=-25, lon_max=45, lat_min=34, lat_max=72):
    """Keep only cells inside a Europe-ish bounding box."""
    return {
        k: (la, lo)
        for k, (la, lo) in centers.items()
        if lat_min <= la <= lat_max and lon_min <= lo <= lon_max
    }

In [ ]:
cell_centers = grid_cell_centers(trace_air)
#europe_cell_centers = filter_europe(cell_centers)
plot_grid_on_worldmap(cell_centers)

## 12. Difference heatmap

Mean temperature difference between two datasets, cell by cell, on a world map.
The four small helpers below do the heavy lifting; `plot_air_diff_heatmap`
assembles the figure.

In [ ]:
def _nearest_idx(coord, queries):
    """Index of the nearest value in 1-D `coord` for each query.

    Handles any coordinate order (lat is often descending). Pure index math —
    reads no data.
    """
    coord = np.asarray(coord, float)
    order = np.argsort(coord)                       # ascending view
    sv = coord[order]
    pos = np.clip(np.searchsorted(sv, queries), 1, len(sv) - 1)
    left_better = np.abs(queries - sv[pos - 1]) <= np.abs(queries - sv[pos])
    return order[np.where(left_better, pos - 1, pos)]


def _time_range_ka(row):
    """(min, max) ka BP of one file — reads the time coord only (cheap)."""
    with xr.open_dataset(row["path"], decode_times=False) as ds:
        ka = np.asarray(time_to_ka_bp(ds["time"]), float)
    return ka.min(), ka.max()


def _extract_point_means(row, q_lats, q_lons, t_lo, t_hi):
    """Mean temp (C) at every (q_lat, q_lon) over [t_lo, t_hi] ka BP.

    One vectorized read of only the requested points, so it scales from the
    tiny PalMod2 file to the 150 GB CHELSA file without materialising the grid.
    """
    with xr.open_dataset(row["path"], decode_times=False) as ds:
        da = ds[row["variable"]]
        ka = np.asarray(time_to_ka_bp(ds["time"]), float)
        da = da.assign_coords(time=("time", ka))

        lat_name = next(n for n in LAT_NAMES if n in da.coords)
        lon_name = next(n for n in LON_NAMES if n in da.coords)

        for d in DEPTH_NAMES:                       # topmost soil layer if present
            if d in da.dims:
                da = da.isel({d: 0})
                break
        if "month" in da.dims:                      # collapse seasonal cycle
            da = da.mean("month")

        # Keep only the shared time window — fewer timesteps to read.
        da = da.isel(time=np.where((ka >= t_lo) & (ka <= t_hi))[0])

        # 0-360 vs -180-180 longitude convention.
        q_lon = np.asarray(q_lons, float).copy()
        if float(da[lon_name].max()) > 180:
            q_lon[q_lon < 0] += 360

        # Vectorized point selection: both indexers share dim "pt" 
        # -> result is (pt, time). This is the single read that touches the big variable.
        lat_i = xr.DataArray(_nearest_idx(da[lat_name].values, q_lats), dims="pt")
        lon_i = xr.DataArray(_nearest_idx(da[lon_name].values, q_lon), dims="pt")
        sub = da.isel({lat_name: lat_i, lon_name: lon_i})

        means = sub.mean("time").values            # data loads here, (pt,)-sized

    return means - 273.15 if row["unit"] == "K" else means


def _cell_edges(centers_1d):
    """Cell-center positions -> cell edges, for pcolormesh on a regular grid."""
    c = np.asarray(centers_1d, dtype=float)
    mids = (c[:-1] + c[1:]) / 2
    return np.concatenate([[2 * c[0] - mids[0]], mids, [2 * c[-1] - mids[-1]]])

In [ ]:
def plot_air_diff_heatmap(centers, df=df_air,
                          name_a="PalMod2", name_b="TraCE-21k",
                          quantity="air", title=None):
    """Heatmap of mean temperature difference (name_a - name_b) per grid cell.

    PRGn: green = name_a warmer, purple = name_a colder, white = equal.
    Works for any `quantity` (air or soil) — the label just follows the arg.
    """
    row_a = df.loc[df["name"] == name_a].iloc[0]
    row_b = df.loc[df["name"] == name_b].iloc[0]

    # Shared time window from coords only (cheap), so both means cover it.
    a_lo, a_hi = _time_range_ka(row_a)
    b_lo, b_hi = _time_range_ka(row_b)
    t_lo, t_hi = max(a_lo, b_lo), min(a_hi, b_hi)

    # Sample exactly the points in `centers` (full grid OR a subset like `europe`); missing cells stay NaN.
    pts = list(centers.values())
    q_lats = np.array([la for la, lo in pts])
    q_lons = np.array([lo for la, lo in pts])

    va = _extract_point_means(row_a, q_lats, q_lons, t_lo, t_hi)
    vb = _extract_point_means(row_b, q_lats, q_lons, t_lo, t_hi)
    d = va - vb                                      # >0 -> name_a warmer -> green

    # Scatter the per-point diffs back onto a regular (lat x lon) grid.
    uniq_lats = sorted({la for la, lo in pts})
    uniq_lons = sorted({lo for la, lo in pts})
    lat_pos = {la: i for i, la in enumerate(uniq_lats)}
    lon_pos = {lo: j for j, lo in enumerate(uniq_lons)}
    diff = np.full((len(uniq_lats), len(uniq_lons)), np.nan)
    for (la, lo), val in zip(pts, d):                # pure-Python, a few hundred floats
        diff[lat_pos[la], lon_pos[lo]] = val

    # Symmetric scale so 0 (no difference) lands exactly on white.
    M = float(np.nanmax(np.abs(diff)))
    if not np.isfinite(M) or M == 0:
        M = 1.0
    norm = TwoSlopeNorm(vmin=-M, vcenter=0.0, vmax=M)

    fig = plt.figure(figsize=(14, 7))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    ax.set_global()
    ax.add_feature(cfeature.LAND,      facecolor="#EAE6DE", zorder=0)
    ax.add_feature(cfeature.OCEAN,     facecolor="#D6E4F0", zorder=0)
    ax.add_feature(cfeature.COASTLINE, lw=0.4, edgecolor="0.4", zorder=2)
    ax.gridlines(lw=0.3, color="0.7", alpha=0.5, draw_labels=True)

    mesh = ax.pcolormesh(
        _cell_edges(uniq_lons), _cell_edges(uniq_lats), diff,
        cmap="PRGn", norm=norm, shading="flat",
        transform=ccrs.PlateCarree(), zorder=1, alpha=0.9,
    )
    cbar = fig.colorbar(mesh, ax=ax, orientation="vertical",
                        shrink=0.7, pad=0.02, extend="both")
    cbar.set_label(f"mean {quantity} temp:  {name_a} - {name_b}  (C)\n"
                   f"green = {name_a} warmer / purple = {name_a} colder")
    ax.set_title(title or
                 f"Mean {quantity}-temperature difference  ({name_a} - {name_b})\n"
                 f"averaged over {t_lo:.1f}-{t_hi:.1f} ka BP, n={len(centers)} cells")
    plt.tight_layout()
    plt.show()
    return diff, uniq_lats, uniq_lons

## 13. Run the difference heatmaps

In [ ]:
# Soil: PalMod2 vs TraCE-21k
diff, dlat, dlon = plot_air_diff_heatmap(cell_centers,
                                         df=df_soil,
                                         name_a="PalMod2",
                                         name_b="TraCE-21k",
                                         quantity="soil",
                                         title=None)

In [ ]:
# Air: every pair of the four datasets.
sets = ["Beyer2020",
        "PalMod2",
        "TraCE-21k",
        "CHELSA-TraCE21k-centennial"]

pairs = list(combinations(sets, 2))

for name_a, name_b in pairs:
    diff, dlat, dlon = plot_air_diff_heatmap(cell_centers,
                                             df=df_air,
                                             name_a=name_a,
                                             name_b=name_b,
                                             quantity="air",
                                             title=None)